<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day12_CV_and_RepeatedKFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [1]:
url = ("https://raw.githubusercontent.com/ryanaxiom/"
       "Applied-ML/main/data/carnegie_data.csv")
edu_url = ("https://raw.githubusercontent.com/ryanaxiom/"
           "Applied-ML/main/data/education.csv")

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
d["med_yn"] = (d["medical"] == 1).astype(int)
y = d["research"]
pile = ["fallenr20", "totdeg", "facnum", "stem_rsd", "med_yn"]
def rmse(a, b): return np.sqrt(np.mean((a - b)**2))

edu = pd.read_csv(edu_url)      # new today: 50 states
spend = edu["spend"]
three = edu[["income", "young", "urban"]]
d.shape, edu.shape

((302, 103), (50, 6))

In [3]:
print(round(rmse(spend, np.full(len(spend), spend.mean())),2))
print(spend.mean())
edu.head(3)

20.72
85.04


,state,spend,income,young,urban,region
0,ME,61,1704,388,399,1
1,NH,68,1885,372,598,1
2,VT,72,1745,397,370,1


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=309)
model = LinearRegression()
oof = cross_val_predict(model, three, spend, cv=kf)
print(round(rmse(spend, oof),2))

16.62


In [6]:
for s in [0,1,2,309]:
  kf = KFold(n_splits=5, shuffle=True, random_state=s)
  e = cross_val_predict(model, three, spend, cv=kf)
  c = cross_val_predict(model, d[pile],y,cv=kf)
  print(s,round(rmse(spend,e),2),round(rmse(y,c)))


0 16.9 201413
1 21.85 209905
2 16.79 201430
309 16.62 201981


In [7]:
vals = []
for s in range(200):
    kf = KFold(n_splits=5, shuffle=True, random_state=s)
    oof = cross_val_predict(model, three, spend, cv=kf)
    vals.append(rmse(spend, oof))
print(round(min(vals), 2), round(max(vals), 2))
print(round((max(vals) - min(vals)) / np.median(vals), 3))

15.57 24.19
0.51


In [8]:
two = edu[["income", "young"]]
for s in [0, 309]:
    kf = KFold(n_splits=5, shuffle=True, random_state=s)
    a = rmse(spend, cross_val_predict(model, two, spend, cv=kf))
    b = rmse(spend, cross_val_predict(model, three, spend, cv=kf))
    print(s, round(a, 2), round(b, 2))

0 16.99 16.9
309 16.49 16.62


In [9]:
# the Carnegie contrast: the Day 9 trio against the pile, same 200 shuffles
trio = ["fallenr20", "med_yn", "stem_rsd"]
margins = []
for s in range(200):
    kf = KFold(n_splits=5, shuffle=True, random_state=s)
    t = rmse(y, cross_val_predict(model, d[trio], y, cv=kf))
    p = rmse(y, cross_val_predict(model, d[pile], y, cv=kf))
    margins.append(t - p)
print(sum(m > 0 for m in margins), "of 200 favour the pile; worst margin", round(min(margins)))

200 of 200 favour the pile; worst margin 28594


In [10]:
a, b = edu.index[edu["state"].isin(["AL", "KY"])]
def same_fold(s):
    kf = KFold(n_splits=5, shuffle=True, random_state=s)
    return any(a in te and b in te for _, te in kf.split(edu))

tog = [v for s, v in enumerate(vals) if same_fold(s)]
apt = [v for s, v in enumerate(vals) if not same_fold(s)]
print(len(tog), round(min(tog), 2), round(max(tog), 2))
print(len(apt), round(min(apt), 2), round(max(apt), 2))

42 20.27 24.19
158 15.57 18.28


In [13]:
def repeated_cv(X, y, seeds):
    out = []
    for s in seeds:
        kf = KFold(n_splits=5, shuffle=True, random_state=s)
        out.append(rmse(y, cross_val_predict(model, X, y, cv=kf)))
    return np.mean(out)
print(round(repeated_cv(three, spend, range(10)), 2))

17.3


In [22]:
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score

rkf = RepeatedKFold(n_splits=5, n_repeats=20, random_state=309)
scores = cross_val_score(model, three, spend, cv=rkf,
                         scoring="neg_mean_squared_error")
print(round(np.sqrt(-scores.mean()), 2))

17.45
